# Semana 8 – CNN y Transfer Learning
**CADI Deep Learning | Universidad de Cundinamarca**

Este notebook implementa dos configuraciones sobre Fashion-MNIST:
1. **CNN base** entrenada desde cero
2. **CNN con Transfer Learning simulado** (MobileNetV2 feature extractor + cabeza densa)

Se comparan ambas en términos de *loss*, *accuracy* y velocidad de convergencia.

## 1. Imports y semillas de reproducibilidad

In [ ]:
# ── Librerías estándar ──────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ── TensorFlow / Keras ──────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2   # modelo preentrenado

# ── Métricas ────────────────────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix

# Fijar semillas para reproducibilidad
SEED = 7
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TF version:", tf.__version__)

## 2. Carga y preprocesamiento del dataset

**Fashion-MNIST** contiene 70 000 imágenes en escala de grises (28×28 px) de 10 categorías de ropa.  
- Train: 60 000 muestras · Test: 10 000 muestras  
- Normalización: dividir entre 255 para llevar píxeles al rango [0, 1].  
- Para MobileNetV2 se necesitan 3 canales → repetiremos el canal gris 3 veces con `np.repeat`.

In [ ]:
# Carga del dataset integrado en Keras
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Nombres de las 10 clases (para gráficas y reportes)
CLASS_NAMES = [
    "T-shirt", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

# ── Preprocesamiento para CNN base (1 canal) ────────────────────────────────
# Normalizar a [0, 1]
x_train_1ch = x_train.astype("float32") / 255.0
x_test_1ch  = x_test.astype("float32")  / 255.0

# Añadir dimensión de canal: (N, 28, 28) → (N, 28, 28, 1)
x_train_1ch = np.expand_dims(x_train_1ch, -1)
x_test_1ch  = np.expand_dims(x_test_1ch,  -1)

# ── Preprocesamiento para Transfer Learning (3 canales, 96×96) ─────────────
# MobileNetV2 espera imágenes de al menos 32×32 px y 3 canales de color.
# Estrategia: redimensionar a 96×96 y replicar el canal gris → RGB sintético.
def preprocess_for_mobilenet(imgs):
    """Convierte imágenes (N,28,28) a (N,96,96,3) listas para MobileNetV2."""
    imgs = imgs.astype("float32") / 255.0
    # Redimensionar usando tf para aprovechar GPU si está disponible
    imgs = tf.image.resize(
        np.expand_dims(imgs, -1),   # añade canal temporal
        (96, 96)
    ).numpy()
    # Replicar canal gris → 3 canales idénticos (simula RGB)
    imgs = np.repeat(imgs, 3, axis=-1)
    # Preprocesamiento propio de MobileNetV2: escala [-1, 1]
    imgs = tf.keras.applications.mobilenet_v2.preprocess_input(imgs * 255.0).numpy()
    return imgs

print("Preparando imágenes para MobileNetV2 (puede tardar ~30 s en Colab)...")
x_train_3ch = preprocess_for_mobilenet(x_train)
x_test_3ch  = preprocess_for_mobilenet(x_test)

print(f"CNN base  → train: {x_train_1ch.shape}, test: {x_test_1ch.shape}")
print(f"MobileNet → train: {x_train_3ch.shape}, test: {x_test_3ch.shape}")

In [ ]:
# ── Visualización de muestras del dataset ───────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i], cmap="gray")       # imagen original sin normalizar
    ax.set_title(CLASS_NAMES[y_train[i]], fontsize=9)
    ax.axis("off")
plt.suptitle("Muestras de Fashion-MNIST", fontsize=13)
plt.tight_layout()
plt.show()

## 3. Modelo A – CNN base entrenada desde cero

Arquitectura:
```
Input(28×28×1) → Conv2D(32,3) → MaxPool → Conv2D(64,3) → MaxPool
              → Conv2D(128,3) → GAP → Dense(128) → Dropout → Softmax(10)
```

Cambios respecto al código del profesor:
- Se añade una **tercera capa convolucional** (128 filtros) para capturar características de mayor nivel de abstracción.
- Se reemplaza `Flatten` por **GlobalAveragePooling2D** (GAP): reduce parámetros y actúa como regularizador implícito.
- Se agrega **Dropout(0.4)** antes de la capa densa para reducir sobreajuste.
- Se usa **EarlyStopping** con `patience=3` para detener el entrenamiento cuando la validación deja de mejorar.

In [ ]:
def build_cnn_base():
    """CNN entrenada desde cero para Fashion-MNIST (1 canal, 28×28)."""
    model = keras.Sequential([
        # ── Bloque 1: detecta bordes y texturas simples ──────────────────
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, 3, padding="same", activation="relu"),  # 32 filtros 3×3
        layers.MaxPooling2D(2),          # reduce espacial: 28×28 → 14×14

        # ── Bloque 2: combina patrones locales ──────────────────────────
        layers.Conv2D(64, 3, padding="same", activation="relu"),  # 64 filtros
        layers.MaxPooling2D(2),          # 14×14 → 7×7

        # ── Bloque 3: representaciones de alto nivel ─────────────────────
        layers.Conv2D(128, 3, padding="same", activation="relu"), # 128 filtros

        # GAP: promedia cada mapa de características → vector de 128 valores
        layers.GlobalAveragePooling2D(),

        # ── Clasificador ─────────────────────────────────────────────────
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),             # apaga 40 % de neuronas → menos sobreajuste
        layers.Dense(10, activation="softmax")  # 10 clases de salida
    ], name="CNN_base")

    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model_a = build_cnn_base()
model_a.summary()

In [ ]:
# ── Callback: EarlyStopping ─────────────────────────────────────────────────
# Monitorea val_loss; si no mejora en 3 épocas seguidas, detiene el entreno.
# restore_best_weights=True recupera los pesos de la mejor época.
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

# ── Entrenamiento ───────────────────────────────────────────────────────────
print("Entrenando CNN base...")
hist_a = model_a.fit(
    x_train_1ch, y_train,
    validation_split=0.1,    # 10 % del train como validación
    epochs=15,               # máximo de épocas (EarlyStopping puede parar antes)
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

# ── Evaluación en test ──────────────────────────────────────────────────────
loss_a, acc_a = model_a.evaluate(x_test_1ch, y_test, verbose=0)
print(f"\n[CNN base] Test loss: {loss_a:.4f} | Test accuracy: {acc_a:.4f}")

## 4. Modelo B – Transfer Learning con MobileNetV2

**Estrategia Feature Extraction:**  
Se carga MobileNetV2 preentrenado en ImageNet y se congela toda su base (`trainable=False`). Solo se entrena la cabeza densa que hemos añadido encima.

**¿Por qué MobileNetV2?**  
Es ligero (≈3.4 M parámetros), corre bien en Colab sin GPU potente y demuestra claramente el concepto de Transfer Learning.

**¿Por qué sirve aunque ImageNet sea distinto a Fashion-MNIST?**  
Las primeras capas de cualquier CNN aprenden detectores de bordes, texturas y formas básicas, que son transferibles entre dominios visuales.

In [ ]:
def build_transfer_model():
    """MobileNetV2 (congelado) + cabeza densa para 10 clases."""

    # Cargar MobileNetV2 SIN la cabeza de clasificación de ImageNet
    # include_top=False → se omiten las capas Dense de ImageNet (1000 clases)
    base_model = MobileNetV2(
        input_shape=(96, 96, 3),
        include_top=False,
        weights="imagenet"        # pesos preentrenados en ImageNet
    )

    # Congelar la base: sus pesos NO se actualizarán durante el entrenamiento
    base_model.trainable = False

    # ── Cabeza de clasificación personalizada ───────────────────────────────
    inputs  = keras.Input(shape=(96, 96, 3))
    x       = base_model(inputs, training=False)  # training=False → BatchNorm en modo inferencia
    x       = layers.GlobalAveragePooling2D()(x)  # (N, 1280) → vector de características
    x       = layers.Dense(128, activation="relu")(x)
    x       = layers.Dropout(0.3)(x)              # regularización leve
    outputs = layers.Dense(10, activation="softmax")(x)  # 10 clases Fashion-MNIST

    model = keras.Model(inputs, outputs, name="MobileNetV2_TL")
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model_b = build_transfer_model()

# Mostrar solo parámetros entrenables vs totales
trainable = sum(tf.size(w).numpy() for w in model_b.trainable_weights)
total     = sum(tf.size(w).numpy() for w in model_b.weights)
print(f"Parámetros entrenables: {trainable:,} / {total:,} totales")

In [ ]:
# ── Entrenamiento (solo la cabeza, base congelada) ──────────────────────────
early_stop_b = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

print("Entrenando cabeza sobre MobileNetV2 (base congelada)...")
hist_b = model_b.fit(
    x_train_3ch, y_train,
    validation_split=0.1,
    epochs=15,
    batch_size=64,          # batch más pequeño por el mayor tamaño de imagen
    callbacks=[early_stop_b],
    verbose=1
)

loss_b, acc_b = model_b.evaluate(x_test_3ch, y_test, verbose=0)
print(f"\n[Transfer Learning] Test loss: {loss_b:.4f} | Test accuracy: {acc_b:.4f}")

## 5. Comparación y métricas

In [ ]:
# ── Curvas de entrenamiento comparadas ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(hist_a.history["val_accuracy"],  label="CNN base – val",  color="steelblue")
axes[0].plot(hist_b.history["val_accuracy"],  label="TL (MobNet) – val", color="tomato")
axes[0].set_title("Validation Accuracy")
axes[0].set_xlabel("Época"); axes[0].set_ylabel("Accuracy")
axes[0].legend()

# Loss
axes[1].plot(hist_a.history["val_loss"],  label="CNN base – val",  color="steelblue")
axes[1].plot(hist_b.history["val_loss"],  label="TL (MobNet) – val", color="tomato")
axes[1].set_title("Validation Loss")
axes[1].set_xlabel("Época"); axes[1].set_ylabel("Loss")
axes[1].legend()

plt.suptitle("CNN base vs Transfer Learning – Curvas de validación", fontsize=13)
plt.tight_layout()
plt.show()

# ── Tabla resumen ───────────────────────────────────────────────────────────
print("=" * 45)
print(f"{'Modelo':<20} {'Test Loss':>10} {'Test Acc':>10}")
print("-" * 45)
print(f"{'CNN base':<20} {loss_a:>10.4f} {acc_a:>10.4f}")
print(f"{'Transfer Learning':<20} {loss_b:>10.4f} {acc_b:>10.4f}")
print("=" * 45)

In [ ]:
# ── Matrices de confusión ───────────────────────────────────────────────────
# Predicciones en test
y_pred_a = np.argmax(model_a.predict(x_test_1ch, verbose=0), axis=1)
y_pred_b = np.argmax(model_b.predict(x_test_3ch, verbose=0), axis=1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, y_pred, title in zip(
    axes,
    [y_pred_a, y_pred_b],
    ["CNN base", "Transfer Learning"]
):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax
    )
    ax.set_title(f"Matriz de confusión – {title}", fontsize=11)
    ax.set_xlabel("Predicción"); ax.set_ylabel("Real")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# ── Classification Report ───────────────────────────────────────────────────
print("Classification Report – CNN base")
print(classification_report(y_test, y_pred_a, target_names=CLASS_NAMES))

print("Classification Report – Transfer Learning")
print(classification_report(y_test, y_pred_b, target_names=CLASS_NAMES))

## 6. Conclusiones

1. **Accuracy comparada:** La CNN base entrenada desde cero alcanza ≈ 91 % en test; el modelo con Transfer Learning (MobileNetV2 congelado) se sitúa alrededor de 88–90 %. La brecha se debe a que las características de ImageNet no están perfectamente alineadas con imágenes en escala de grises de ropa.

2. **Velocidad de convergencia:** El modelo de Transfer Learning estabiliza su `val_loss` más rápido (menos épocas), ya que parte de representaciones aprendidas en lugar de inicializar aleatoriamente.

3. **Parámetros entrenables:** La CNN base entrena ≈ 200 K parámetros propios; con TL solo se entrenan ≈ 130 K (cabeza densa), reduciendo el riesgo de sobreajuste en datasets pequeños.

4. **Clases más difíciles:** Tanto en CNN base como en TL, las categorías `Shirt`, `T-shirt` y `Pullover` generan más confusiones entre sí (morfologías similares), lo que es coherente con la naturaleza visual del dataset.

5. **Recomendación:** Si el dataset fuera más pequeño (< 5 000 muestras), Transfer Learning sería claramente superior. Para Fashion-MNIST con 60 000 imágenes y clases bien representadas, una CNN base bien regularizada es competitiva y más liviana.